# RAG Evaluation Metrics [Step 1 - Faithfulness, Relevance, Context Precision]

> **MLCourse - Agentic AI - RAG Evaluation**

Evaluating a RAG system requires more than just checking if the answer
"sounds right." We need to measure three core dimensions:

1. **Faithfulness**: Is every claim in the answer supported by the retrieved context?
2. **Answer Relevance**: Does the answer actually address the question asked?
3. **Context Precision**: Are the retrieved chunks actually useful and ranked correctly?

This notebook defines and implements each metric from scratch using simple
Python and an LLM-as-judge approach. No external evaluation frameworks needed.

### 1. Setup and Environment


In [2]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

print("[setup] Evaluation metrics notebook initialized")

[setup] Evaluation metrics notebook initialized


### 2. Initialize the LLM (used as judge)


In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("[llm] ChatOllama ready:", llm.model)

[llm] ChatOllama ready: llama3.1:8b


### 3. Build a Simple RAG Pipeline (test subject)


In [6]:
from pathlib import Path
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

ALICE_PATH = Path(r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt")
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")[:25_000]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_text(raw_text)
documents = [
    Document(page_content=c, metadata={"chunk_id": i})
    for i, c in enumerate(chunks)
]
print(f"[rag] Loaded {len(documents)} chunks from Alice in Wonderland")

[rag] Loaded 77 chunks from Alice in Wonderland


### Build the vector store and retriever.


In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"[rag] FAISS index built with {vectorstore.index.ntotal} vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[rag] FAISS index built with 77 vectors


### Define the RAG chain: retrieve -> format -> generate.


In [10]:
def format_docs(docs):
    """Format documents into a single string with chunk markers."""
    parts = []
    for i, doc in enumerate(docs):
        parts.append(f"[Chunk {doc.metadata.get('chunk_id', i)}] {doc.page_content}")
    return "\n\n".join(parts)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about Alice in Wonderland using ONLY the provided context. "
     "Cite chunk numbers like [Chunk 3] when referencing specific information. "
     "If the context does not contain enough information, say so."),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("[rag] RAG chain ready")

[rag] RAG chain ready


### 4. Create Evaluation Data


In [12]:
eval_data = [
    {
        "question": "Why did the White Rabbit run down the hole?",
        "reference_answer": "The White Rabbit ran down the hole because he was in a hurry and running late.",
        "source_chunks": ["He ran down the hole because he was in a great hurry."],
    },
    {
        "question": "What did Alice find on the table?",
        "reference_answer": "Alice found a small golden key on a tiny glass table, and a door behind the wall about fifteen inches high.",
        "source_chunks": ["she found a tiny golden key", "a little door about fifteen inches high"],
    },
    {
        "question": "Who was the Queen of Hearts?",
        "reference_answer": "The Queen of Hearts was a tyrannical ruler who ordered people to be beheaded for minor offenses.",
        "source_chunks": ["Off with his head!", "The Queen of Hearts"],
    },
]

print(f"[eval] Created {len(eval_data)} evaluation examples")

[eval] Created 3 evaluation examples


### 5. Metric 1: Faithfulness


In [14]:
def extract_claims(answer: str) -> list:
    """Extract individual factual claims from an answer using the LLM."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Extract each individual factual claim from the following answer. "
         "Return them as a numbered list. Each claim should be a single "
         "verifiable statement. Example:\n"
         "1. The rabbit wore a waistcoat.\n"
         "2. The rabbit carried a pocket watch."),
        ("user", "Answer: {answer}")
    ])
    response = (prompt | llm | StrOutputParser()).invoke({"answer": answer})

    claims = []
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            # Remove leading number and dot/paren
            claim = re.sub(r"^\d+[\.\)]\s*", "", line)
            if claim:
                claims.append(claim)
    return claims


def check_claim_support(claim: str, context: str) -> bool:
    """Check if a single claim is supported by the context."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a fact checker. Determine if the following claim is "
         "SUPPORTED or NOT SUPPORTED by the context. Reply with exactly "
         "one word: SUPPORTED or NOT SUPPORTED."),
        ("user", "Context:\n{context}\n\nClaim: {claim}")
    ])
    response = (prompt | llm | StrOutputParser()).invoke({
        "context": context, "claim": claim
    })
    return "SUPPORTED" in response.upper()


def compute_faithfulness(answer: str, context: str) -> dict:
    """Compute faithfulness score: fraction of claims supported by context."""
    claims = extract_claims(answer)
    if not claims:
        return {"score": 0.0, "claims": [], "message": "No claims extracted"}

    supported = 0
    results = []
    for claim in claims:
        is_supported = check_claim_support(claim, context)
        results.append({"claim": claim, "supported": is_supported})
        if is_supported:
            supported += 1

    score = supported / len(claims)
    return {"score": score, "claims": results, "total": len(claims),
            "supported_count": supported}

### Test the faithfulness metric on a sample query.


In [16]:
sample_query = "What did the White Rabbit say when he arrived?"

# Get the RAG answer
rag_answer = rag_chain.invoke(sample_query)
# Get the retrieved context
sample_docs = retriever.invoke(sample_query)
sample_context = format_docs(sample_docs)

print(f"Query: {sample_query}")
print(f"Answer: {rag_answer[:200]}...")
print()

faithfulness = compute_faithfulness(rag_answer, sample_context)
print(f"Faithfulness Score: {faithfulness['score']:.2f}")
print(f"Claims checked: {faithfulness['total']}")
print(f"Supported: {faithfulness['supported_count']}")
for item in faithfulness["claims"]:
    status = "Y" if item["supported"] else "N"
    print(f"  [{status}] {item['claim'][:80]}")

Query: What did the White Rabbit say when he arrived?
Answer: The White Rabbit said "Oh dear! Oh dear! I shall be late!" [Chunk 4]....



Faithfulness Score: 1.00
Claims checked: 4
Supported: 4
  [Y] The White Rabbit spoke.
  [Y] The White Rabbit said "Oh dear!"
  [Y] The White Rabbit said "I shall be late."
  [Y] The White Rabbit was concerned about being late.


### 6. Metric 2: Answer Relevance


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# We reuse the same embeddings model for similarity
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


def generate_questions(answer: str, n: int = 3) -> list:
    """Generate questions that the answer could be responding to."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Generate {n} questions that the following answer could be responding to. "
         "Return them as a numbered list."),
        ("user", "Answer: {answer}")
    ])
    response = (prompt | llm | StrOutputParser()).invoke({"answer": answer, "n": n})

    questions = []
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            q = re.sub(r"^\d+[\.\)]\s*", "", line)
            if q:
                questions.append(q)
    return questions[:n]


def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors."""
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)


def compute_answer_relevance(question: str, answer: str) -> dict:
    """Compute answer relevance by generating questions from the answer
    and measuring their similarity to the original question."""
    generated_qs = generate_questions(answer)
    if not generated_qs:
        return {"score": 0.0, "message": "No questions generated"}

    # Embed all questions
    all_texts = [question] + generated_qs
    embeddings_list = embed_model.embed_documents(all_texts)
    original_emb = embeddings_list[0]
    generated_embs = embeddings_list[1:]

    # Average similarity of generated questions to original
    similarities = [cosine_similarity(original_emb, gen_emb)
                    for gen_emb in generated_embs]
    avg_score = float(np.mean(similarities))

    return {"score": avg_score, "original_question": question,
            "generated_questions": generated_qs,
            "similarities": similarities}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Test the answer relevance metric.


In [20]:
relevance = compute_answer_relevance(sample_query, rag_answer)
print(f"Answer Relevance Score: {relevance['score']:.4f}")
print(f"Original question: {relevance['original_question']}")
print(f"Generated questions from answer:")
for i, q in enumerate(relevance["generated_questions"]):
    sim = relevance["similarities"][i]
    print(f"  {i+1}. (sim={sim:.4f}) {q}")

Answer Relevance Score: 0.7924
Original question: What did the White Rabbit say when he arrived?
Generated questions from answer:
  1. (sim=0.9347) What did the White Rabbit say when he was in a hurry?
  2. (sim=0.7124) In Chapter 4 of the story, what phrase is associated with the White Rabbit's anxiety?
  3. (sim=0.7300) How does the White Rabbit react when he realizes he will be late?


### 7. Metric 3: Context Precision


In [22]:
def is_chunk_relevant(question: str, chunk: str) -> bool:
    """Determine if a retrieved chunk is relevant to the question."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Determine if the following text passage contains information that "
         "would help answer the question. Reply with exactly one word: "
         "RELEVANT or NOT RELEVANT."),
        ("user", "Question: {question}\n\nPassage: {chunk}")
    ])
    response = (prompt | llm | StrOutputParser()).invoke({
        "question": question, "chunk": chunk
    })
    return "RELEVANT" in response.upper()


def compute_context_precision(question: str, chunks: list) -> dict:
    """Compute context precision: are relevant chunks retrieved and ranked well?

    Uses Average Precision (AP) across the ranked list of chunks.
    AP = (1/R) * SUM(k=1..N) [precision@k * is_relevant(k)]
    where R = total number of relevant chunks in the list.
    """
    relevance_flags = []
    for chunk in chunks:
        is_rel = is_chunk_relevant(question, chunk.page_content)
        relevance_flags.append(is_rel)

    # Compute Average Precision
    relevant_count = sum(relevance_flags)
    if relevant_count == 0:
        return {"score": 0.0, "chunk_relevance": relevance_flags,
                "relevant_count": 0, "total_chunks": len(chunks)}

    running_relevant = 0
    ap_sum = 0.0
    for k, is_rel in enumerate(relevance_flags, start=1):
        if is_rel:
            running_relevant += 1
            precision_at_k = running_relevant / k
            ap_sum += precision_at_k

    ap = ap_sum / relevant_count

    return {"score": ap, "chunk_relevance": relevance_flags,
            "relevant_count": relevant_count,
            "total_chunks": len(chunks),
            "precision_at_k": []}

### Test context precision with the retrieved chunks.


In [24]:
precision = compute_context_precision(sample_query, sample_docs)
print(f"Context Precision (AP): {precision['score']:.4f}")
print(f"Relevant chunks: {precision['relevant_count']}/{precision['total_chunks']}")
for i, (doc, is_rel) in enumerate(zip(sample_docs, precision["chunk_relevance"])):
    status = "RELEVANT" if is_rel else "NOT RELEVANT"
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"  [{status}] Chunk {i}: {preview}...")

Context Precision (AP): 1.0000
Relevant chunks: 4/4
  [RELEVANT] Chunk 0: so desperate that she was ready to ask help of any one; so, when the Rabbit came...
  [RELEVANT] Chunk 1: After a time she heard a little pattering of feet in the distance, and she hasti...
  [RELEVANT] Chunk 2: As she said this she looked down at her hands, and was surprised to see that she...
  [RELEVANT] Chunk 3: There was nothing so _very_ remarkable in that; nor did Alice think it so _very_...


### 8. Combined Evaluation Score


In [26]:
def evaluate_rag_response(question: str, rag_chain, retriever) -> dict:
    """Run the full evaluation suite on a single RAG response."""
    # Get RAG answer and context
    answer = rag_chain.invoke(question)
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # Compute each metric
    faith = compute_faithfulness(answer, context)
    relev = compute_answer_relevance(question, answer)
    prec = compute_context_precision(question, docs)

    # Combined score (equal weights)
    combined = (faith["score"] + relev["score"] + prec["score"]) / 3.0

    return {
        "question": question,
        "answer": answer[:300],
        "faithfulness": faith["score"],
        "answer_relevance": relev["score"],
        "context_precision": prec["score"],
        "combined_score": combined,
    }

### Run the full evaluation on one test question.


In [28]:
result = evaluate_rag_response(
    "What happened when Alice fell down the rabbit hole?",
    rag_chain, retriever
)

print("=" * 60)
print(f"Question: {result['question']}")
print(f"Answer: {result['answer'][:150]}...")
print("-" * 60)
print(f"Faithfulness:      {result['faithfulness']:.4f}")
print(f"Answer Relevance:  {result['answer_relevance']:.4f}")
print(f"Context Precision: {result['context_precision']:.4f}")
print(f"Combined Score:    {result['combined_score']:.4f}")
print("=" * 60)

Question: What happened when Alice fell down the rabbit hole?
Answer: According to [Chunk 6], Alice fell down the rabbit hole after chasing the White Rabbit. The rabbit-hole dipped suddenly down, and Alice had not a mome...
------------------------------------------------------------
Faithfulness:      1.0000
Answer Relevance:  0.8906
Context Precision: 1.0000
Combined Score:    0.9635


### 9. Batch Evaluation


In [30]:
batch_questions = [
    "What did Alice find on the table?",
    "Who was the Cheshire Cat?",
    "What game did they play with flamingos?",
    "Why did Alice grow larger in the house?",
]

print("Batch Evaluation Results")
print("=" * 70)

all_results = []
for q in batch_questions:
    res = evaluate_rag_response(q, rag_chain, retriever)
    all_results.append(res)
    print(f"Q: {q[:50]}")
    print(f"  Faith={res['faithfulness']:.2f}  Rel={res['answer_relevance']:.2f}  Prec={res['context_precision']:.2f}  Combined={res['combined_score']:.2f}")
    print()

# Aggregate
avg_faith = sum(r["faithfulness"] for r in all_results) / len(all_results)
avg_relev = sum(r["answer_relevance"] for r in all_results) / len(all_results)
avg_prec = sum(r["context_precision"] for r in all_results) / len(all_results)
avg_combined = sum(r["combined_score"] for r in all_results) / len(all_results)

print("-" * 70)
print(f"Average Faithfulness:      {avg_faith:.4f}")
print(f"Average Answer Relevance:  {avg_relev:.4f}")
print(f"Average Context Precision: {avg_prec:.4f}")
print(f"Average Combined Score:    {avg_combined:.4f}")

Batch Evaluation Results


Q: What did Alice find on the table?
  Faith=1.00  Rel=0.50  Prec=1.00  Combined=0.83



Q: Who was the Cheshire Cat?
  Faith=0.00  Rel=0.59  Prec=1.00  Combined=0.53



Q: What game did they play with flamingos?
  Faith=0.00  Rel=0.72  Prec=1.00  Combined=0.57



Q: Why did Alice grow larger in the house?
  Faith=1.00  Rel=0.68  Prec=1.00  Combined=0.89

----------------------------------------------------------------------
Average Faithfulness:      0.5000
Average Answer Relevance:  0.6228
Average Context Precision: 1.0000
Average Combined Score:    0.7076


### 10. Metric Comparison Table


In [32]:
print("\nMetric Definitions Summary")
print("=" * 60)
print()
print("Faithfulness")
print("  Measures: Are all claims in the answer supported by context?")
print("  Method: Extract claims, check each against context")
print("  Range: 0.0 (no claims supported) to 1.0 (all supported)")
print()
print("Answer Relevance")
print("  Measures: Does the answer address the question?")
print("  Method: Generate questions from answer, measure similarity")
print("  Range: 0.0 (irrelevant) to 1.0 (highly relevant)")
print()
print("Context Precision")
print("  Measures: Are retrieved chunks relevant and well-ranked?")
print("  Method: Judge chunk relevance, compute Average Precision")
print("  Range: 0.0 (no relevant chunks) to 1.0 (all relevant, well-ranked)")
print()
print("These three metrics form the foundation of RAG evaluation.")
print("In the next notebook we will use the RAGAS framework to automate this process.")


Metric Definitions Summary

Faithfulness
  Measures: Are all claims in the answer supported by context?
  Method: Extract claims, check each against context
  Range: 0.0 (no claims supported) to 1.0 (all supported)

Answer Relevance
  Measures: Does the answer address the question?
  Method: Generate questions from answer, measure similarity
  Range: 0.0 (irrelevant) to 1.0 (highly relevant)

Context Precision
  Measures: Are retrieved chunks relevant and well-ranked?
  Method: Judge chunk relevance, compute Average Precision
  Range: 0.0 (no relevant chunks) to 1.0 (all relevant, well-ranked)

These three metrics form the foundation of RAG evaluation.
In the next notebook we will use the RAGAS framework to automate this process.


### Summary
